In [1]:
import numpy 
import qiskit 
import qsiht_library

import qsiht_library
import numpy as np

angles = np.random.randn(63)
circuit = qsiht_library.generalFastPath(6, angles)

decomposed = circuit.decompose()
print(f"After decompose - Total gates: {decomposed.size()}")
print(decomposed.count_ops())

# Or count manually
count = 0
for instruction in circuit.data:
    count += 1
    print(f"Instruction {count}: {instruction.operation.name}")

After decompose - Total gates: 63
OrderedDict({'c5ry_o0': 6, 'c5ry_o1': 5, 'c5ry_o3': 4, 'c5ry_o2': 4, 'c5ry_o7': 3, 'c5ry_o6': 3, 'c5ry_o5': 3, 'c5ry_o4': 3, 'c5ry_o15': 2, 'c5ry_o14': 2, 'c5ry_o13': 2, 'c5ry_o12': 2, 'c5ry_o11': 2, 'c5ry_o10': 2, 'c5ry_o9': 2, 'c5ry_o8': 2, 'c5ry': 1, 'c5ry_o30': 1, 'c5ry_o29': 1, 'c5ry_o28': 1, 'c5ry_o27': 1, 'c5ry_o26': 1, 'c5ry_o25': 1, 'c5ry_o24': 1, 'c5ry_o23': 1, 'c5ry_o22': 1, 'c5ry_o21': 1, 'c5ry_o20': 1, 'c5ry_o19': 1, 'c5ry_o18': 1, 'c5ry_o17': 1, 'c5ry_o16': 1})
Instruction 1: circuit-42_dg
Instruction 2: circuit-54_dg
Instruction 3: circuit-74_dg
Instruction 4: circuit-110_dg
Instruction 5: circuit-178_dg
Instruction 6: circuit-310_dg


In [12]:
import numpy as np
import qsiht_library
from qiskit import transpile

angles = np.random.randn(64)
circuit = qsiht_library.generalFastPath(4, angles)

# Original depth
print("Depth:", circuit.depth())
print("Gate counts:", circuit.count_ops())
print("Total gates:", sum(circuit.count_ops().values()))

# Decomposed
qc_transpiled = transpile(circuit, basis_gates=['cx', 'ry', 'rz', 'h', 'x'], optimization_level=0)
print("\nAfter transpile:")
print("Depth:", qc_transpiled.depth())
print("Gate counts:", qc_transpiled.count_ops())
print("Total gates:", sum(qc_transpiled.count_ops().values()))

Depth: 4
Gate counts: OrderedDict({'circuit-2507_dg': 1, 'circuit-2513_dg': 1, 'circuit-2521_dg': 1, 'circuit-2533_dg': 1})
Total gates: 4

After transpile:
Depth: 2127
Gate counts: OrderedDict({'rz': 1785, 'h': 420, 'cx': 300, 'x': 56})
Total gates: 2561


In [17]:
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# ── reuse your already-built circuit ─────────────────────────────────────────
# qc_opt or qc_transpiled — whichever you want to benchmark
TARGET = qc_transpiled   # <-- change this if needed

def ensure_measured(qc):
    if qc.num_clbits == 0:
        qc = qc.copy()
        qc.measure_all()
    return qc

def build_noise_model(p):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx'])
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 1), ['ry', 'rz', 'h', 'x'])
    return nm

def run_counts(qc, sim, shots):
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

def fidelity(ideal, noisy, shots):
    keys = set(ideal) | set(noisy)
    return sum(np.sqrt(ideal.get(k,0) * noisy.get(k,0)) for k in keys) / shots

def tv_distance(ideal, noisy, shots):
    keys = set(ideal) | set(noisy)
    return 0.5 * sum(abs(ideal.get(k,0) - noisy.get(k,0)) for k in keys) / shots

# ── sweep parameters — edit freely ───────────────────────────────────────────
shot_counts  = [1_000, 10_000, 100_000, 1_000_000]
noise_levels = [0.001, 0.005, 0.01, 0.02]

# ── run ───────────────────────────────────────────────────────────────────────
qc_m     = ensure_measured(TARGET)
ideal_sim = AerSimulator()

print(f"{'Shots':<12} {'Noise':>8} {'Fidelity':>10} {'TV dist':>10} {'Top state %':>12}")
print("-" * 56)

for shots in shot_counts:
    ideal_counts = run_counts(qc_m, ideal_sim, shots)
    top_state    = max(ideal_counts, key=ideal_counts.get)

    for noise in noise_levels:
        noisy_sim    = AerSimulator(noise_model=build_noise_model(noise))
        noisy_counts = run_counts(qc_m, noisy_sim, shots)

        fid     = fidelity(ideal_counts, noisy_counts, shots)
        tv      = tv_distance(ideal_counts, noisy_counts, shots)
        top_pct = noisy_counts.get(top_state, 0) / shots

        print(f"{shots:<12,} {noise:>8.3f} {fid:>10.4f} {tv:>10.4f} {top_pct:>11.3%}")

    print()  # blank line between shot groups

Shots           Noise   Fidelity    TV dist  Top state %
--------------------------------------------------------
1,000           0.001     0.9499     0.2040     57.100%
1,000           0.005     0.7460     0.5820     20.500%
1,000           0.010     0.6141     0.6850      9.000%
1,000           0.020     0.5626     0.7130      6.000%

10,000          0.001     0.9524     0.2192     56.470%
10,000          0.005     0.7507     0.5787     20.630%
10,000          0.010     0.6321     0.6848     10.090%
10,000          0.020     0.5720     0.7217      6.680%

100,000         0.001     0.9504     0.2132     56.558%
100,000         0.005     0.7448     0.5813     20.368%
100,000         0.010     0.6264     0.6861      9.985%
100,000         0.020     0.5723     0.7141      6.855%

1,000,000       0.001     0.9507     0.2131     56.478%
1,000,000       0.005     0.7439     0.5814     20.326%
1,000,000       0.010     0.6275     0.6846     10.099%
1,000,000       0.020     0.5704     0.7144